# Faza 3 - rewizja filtra na pełnym wariancie D (10 klas)

Cel: na pełnym zbiorze 300/klasę × 10 klas zobaczyć rozkład CLIP/DINOv2 similarity, zdecydować czy progi z kalibracji (`floor_clip=0.65`, `floor_dinov2=0.55`) są OK czy podnieść. Produkuje manifest 'kept' do użycia w treningu.

**Na dysku**
- `data/synthetic/variant_D/manifest_kept.csv` - lista obrazów które przeszły filtr (per breed)
- `data/synthetic/variant_D/filter_scores.csv` - clip_sim, dino_sim, kept per obraz (do dalszej analizy)
- `data/synthetic/variant_D/decision_v2.json` - finalne progi + retention per klasa
- Inline: histogramy per klasa, tabele retention przy różnych progach

## 1. Klonowanie + install

In [ ]:
import os
REPO_URL = 'https://github.com/micwuj/DLICV.git'
REPO_DIR = '/content/dlicv'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
!pip install -q transformers==4.46.3 timm==1.0.11 PyYAML==6.0.2 huggingface_hub==0.26.2

## 2. HF login + pull `variant_D/` z HF

In [ ]:
from huggingface_hub import login, snapshot_download, whoami

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except (ImportError, Exception):
    HF_TOKEN = os.environ.get('HF_TOKEN')

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('logged in as:', whoami()['name'])

HF_REPO_ID = 'micwuj/dlicv-synth'
snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type='dataset',
    allow_patterns='variant_D/**',
    local_dir='data/synthetic',
)

## 3. Pobranie Pets (do real refs)

In [ ]:
from pathlib import Path
PETS = Path('data/raw/oxford-iiit-pet/images')
if not (PETS.exists() and any(PETS.iterdir())):
    !python scripts/download_pets.py
else:
    print(f'present: {PETS}')

## 4. Setup

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.synth.filter import FilterConfig, compute_embeddings, nearest_sim
from src.synth.prompts import BREEDS
from src.utils.device import device_info, get_device

device = get_device()
print('device:', device_info(device))

SYNTH_ROOT = Path('data/synthetic/variant_D')
SPLIT_PATH = Path('data/splits/pets_10cls_30perclass_seed0.json')
PETS_IMAGES = Path('data/raw/oxford-iiit-pet/images')

with open(SPLIT_PATH) as f:
    split = json.load(f)
class_to_idx = split['meta']['class_to_idx']
breeds = list(BREEDS.keys())
print('breeds:', breeds)

## 5. Liczenie embeddingów (CLIP + DINOv2) dla wszystkich 10 klas

In [ ]:
fc = FilterConfig()

synth_paths = {b: sorted((SYNTH_ROOT / b).glob(f'{b}_*.png')) for b in breeds}
real_paths = {}
for b in breeds:
    label = class_to_idx[b]
    real_paths[b] = [PETS_IMAGES / e['image'] for e in split['train'] if e['label'] == label]

for b in breeds:
    print(f'{b}: synth={len(synth_paths[b])}, real={len(real_paths[b])}')

embeddings = {}
for kind, mid in [('clip', fc.clip_model_id), ('dinov2', fc.dinov2_model_id)]:
    for b in breeds:
        for tag, paths in [('synth', synth_paths[b]), ('real', real_paths[b])]:
            key = (b, kind, tag)
            embeddings[key] = compute_embeddings(paths, kind, mid, device, batch_size=16)
            print(f'{key}: {tuple(embeddings[key].shape)}')

## 6. Similarity scores per obraz

Dla synth: top-K nearest do real refs (per breed).
Dla real: leave-one-out top-K (in-distribution baseline).

In [ ]:
def loo_top_k(emb: torch.Tensor, k: int) -> torch.Tensor:
    sim = emb @ emb.T
    sim.fill_diagonal_(-1.0)
    k = min(k, emb.shape[0] - 1)
    topk, _ = sim.topk(k=k, dim=1)
    return topk.mean(dim=1)

scores = {}
for b in breeds:
    for kind in ('clip', 'dinov2'):
        real = embeddings[(b, kind, 'real')]
        synth = embeddings[(b, kind, 'synth')]
        scores[(b, kind, 'synth')] = nearest_sim(synth, real, top_k=fc.top_k).numpy()
        scores[(b, kind, 'real')] = loo_top_k(real, k=fc.top_k).numpy()

print('scores ready')

## 7. Histogramy: synth vs real per klasa

In [ ]:
fig, axes = plt.subplots(10, 2, figsize=(11, 24))
for r, b in enumerate(breeds):
    for c, kind in enumerate(['clip', 'dinov2']):
        ax = axes[r, c]
        ax.hist(scores[(b, kind, 'real')], bins=12, alpha=0.6, color='tab:green', label='real LOO')
        ax.hist(scores[(b, kind, 'synth')], bins=20, alpha=0.6, color='tab:orange', label='synth')
        ax.set_title(f'{b} - {kind}', fontsize=10)
        if r == 0:
            ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 8. Tabela: synth statystyki per klasa

In [ ]:
stats_rows = []
for b in breeds:
    for kind in ('clip', 'dinov2'):
        s = scores[(b, kind, 'synth')]
        r = scores[(b, kind, 'real')]
        stats_rows.append({
            'breed': b,
            'kind': kind,
            'synth_median': round(np.median(s), 3),
            'synth_p25': round(np.percentile(s, 25), 3),
            'synth_min': round(s.min(), 3),
            'real_median': round(np.median(r), 3),
            'real_p25': round(np.percentile(r, 25), 3),
        })
stats = pd.DataFrame(stats_rows)
stats_pivot = stats.pivot_table(index='breed', columns='kind', values=['synth_median', 'synth_p25', 'real_median', 'real_p25'])
stats_pivot

## 9. Sweep progów - retention per klasa

In [ ]:
rows = []
for fc_floor in [0.60, 0.65, 0.70, 0.72, 0.75]:
    for fd_floor in [0.45, 0.50, 0.55, 0.60]:
        for b in breeds:
            c = scores[(b, 'clip', 'synth')]
            d = scores[(b, 'dinov2', 'synth')]
            kept = int(((c >= fc_floor) & (d >= fd_floor)).sum())
            rows.append({'breed': b, 'floor_clip': fc_floor, 'floor_dinov2': fd_floor, 'kept': kept, 'kept_pct': round(100 * kept / len(c), 1)})
sweep = pd.DataFrame(rows)

# Per-threshold summary (across all classes)
summary = sweep.groupby(['floor_clip', 'floor_dinov2']).agg(
    min_kept_pct=('kept_pct', 'min'),
    median_kept_pct=('kept_pct', 'median'),
    max_kept_pct=('kept_pct', 'max'),
    worst_breed=('kept_pct', lambda x: sweep.loc[x.idxmin(), 'breed']),
).round(1)
summary

In [ ]:
# Pelna macierz retention per breed dla wybranych progów (łatwiej zobaczyć outliery)
sweep_pivot = sweep[sweep['floor_clip'].isin([0.65, 0.70, 0.72])].pivot_table(
    index=['breed'], columns=['floor_clip', 'floor_dinov2'], values='kept_pct'
)
sweep_pivot

## 10. Decyzja + manifest 'kept'

Po obejrzeniu sekcji 8-9, ustaw `PROPOSED_FLOOR_*`. Komórka zapisze manifest do treningu i decision_v2.json.

In [ ]:
PROPOSED_FLOOR_CLIP = 0.65
PROPOSED_FLOOR_DINOV2 = 0.55

manifest_rows = []
score_rows = []
retention = {}
for b in breeds:
    c = scores[(b, 'clip', 'synth')]
    d = scores[(b, 'dinov2', 'synth')]
    mask = (c >= PROPOSED_FLOOR_CLIP) & (d >= PROPOSED_FLOOR_DINOV2)
    retention[b] = {'kept': int(mask.sum()), 'total': len(c)}
    for path, cs, ds, kept in zip(synth_paths[b], c, d, mask):
        rel = path.as_posix()  # already starts with data/synthetic/variant_D/...
        score_rows.append({'breed': b, 'path': rel, 'clip_sim': float(cs), 'dinov2_sim': float(ds), 'kept': bool(kept)})
        if kept:
            manifest_rows.append({'breed': b, 'path': rel})

scores_df = pd.DataFrame(score_rows)
manifest_df = pd.DataFrame(manifest_rows)
scores_df.to_csv(SYNTH_ROOT / 'filter_scores.csv', index=False)
manifest_df.to_csv(SYNTH_ROOT / 'manifest_kept.csv', index=False)

decision = {
    'floor_clip': PROPOSED_FLOOR_CLIP,
    'floor_dinov2': PROPOSED_FLOOR_DINOV2,
    'top_k': fc.top_k,
    'variant': 'D',
    'breeds': breeds,
    'retention_per_breed': retention,
    'total_kept': int(manifest_df.shape[0]),
    'total_generated': int(scores_df.shape[0]),
}
(SYNTH_ROOT / 'decision_v2.json').write_text(json.dumps(decision, indent=2))

print(f"kept {decision['total_kept']}/{decision['total_generated']} ({100*decision['total_kept']/decision['total_generated']:.1f}%)")
for b, r in retention.items():
    print(f"  {b}: {r['kept']}/{r['total']}")
print(f'\nsaved: filter_scores.csv, manifest_kept.csv, decision_v2.json')

## 11. Push manifestu na HF

In [ ]:
from huggingface_hub import HfApi

if HF_TOKEN:
    api = HfApi()
    for fname in ['filter_scores.csv', 'manifest_kept.csv', 'decision_v2.json']:
        api.upload_file(
            path_or_fileobj=str(SYNTH_ROOT / fname),
            path_in_repo=f'variant_D/{fname}',
            repo_id=HF_REPO_ID,
            repo_type='dataset',
            commit_message=f'filter review v2: {fname}',
        )
    print(f'uploaded -> https://huggingface.co/datasets/{HF_REPO_ID}/tree/main/variant_D')
else:
    print('skip: no HF_TOKEN')